# 2차 — 환자(추정) 그룹 기반 Target Encoding

## 배경
환자 단위 누수 진단 때 발견한 것: "안정적 특성"(나이, 불임원인 플래그, 난자/정자 출처 등 22개
컬럼) 조합이 같은 행들의 그룹에서, 그룹이 커질수록 "총 시술 횟수가 전부 다른 값"인 비율이
무작위 기준선보다 확실히 높았음(그룹크기 6: 관측 4.8% vs 무작위 0.9%) → **train에 같은 환자의
반복 기록이 실제로 존재한다**는 통계적 증거.

그땐 진단만 했고 피처로는 안 써봤음. 이번엔 "비슷한 프로필의 다른 행들이 어떻게 됐는지"를
target encoding으로 만들어서 실제로 도움이 되는지 테스트.

## 주의할 점
- `총 임신 횟수`, `총 출산 횟수`, `IVF/DI 임신·출산 횟수` 등이 이미 "이 환자의 과거 성공 이력"을
  직접 알려주고 있어서, 새로 만드는 피처가 **재탕일 가능성이 높음** — 기대치를 너무 높게 잡지 말 것
- **누수 방지가 핵심**: 같은 "환자 추정 그룹"이 train/val에 걸쳐 나뉠 수 있으므로, target encoding은
  **반드시 fold의 train 부분에서만** 계산 (leave-one-out, val 행은 train 그룹평균을 그대로 받기만 함)

## 그룹 정의
나이, 7개 주/부 불임원인 플래그, 9개 "불임원인-X" 플래그, 난자/정자 출처, 난자/정자 기증자 나이
(총 22개 컬럼) — 환자 누수 진단 때 썼던 것과 동일.

---
**저장 위치:** `experiment_history/2차/2차_patient_group_target_encoding.ipynb`


In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings("ignore")


In [2]:
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")
TRAIN_PATH = DATA_DIR / "train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]

ROBUST_PARAMS_PATH = SHARED_DIR / "lgbm_robust_best_params.json"
FALLBACK_PARAMS_PATH = SHARED_DIR / "best_params.json"

N_SPLITS = 5
SEEDS_STAGE1 = [1004, 42, 7]
SEEDS_STAGE2 = [1004, 42, 7, 123, 2024, 88, 555, 999, 31, 77]
STAGE2_PROMOTION_THRESHOLD = 0.0002

TE_SMOOTHING = 20  # 그룹이 작을수록 전체 평균 쪽으로 더 당겨짐 (베이지안 스무딩)

STABLE_COLS = [
    "시술 당시 나이",
    "남성 주 불임 원인", "남성 부 불임 원인", "여성 주 불임 원인", "여성 부 불임 원인",
    "부부 주 불임 원인", "부부 부 불임 원인", "불명확 불임 원인",
    "불임 원인 - 난관 질환", "불임 원인 - 남성 요인", "불임 원인 - 배란 장애",
    "불임 원인 - 여성 요인", "불임 원인 - 자궁경부 문제", "불임 원인 - 자궁내막증",
    "불임 원인 - 정자 농도", "불임 원인 - 정자 면역학적 요인", "불임 원인 - 정자 운동성",
    "불임 원인 - 정자 형태",
    "난자 출처", "정자 출처", "난자 기증자 나이", "정자 기증자 나이",
]


## 1. 데이터 로드 + 전처리 + 환자 추정 그룹 시그니처 생성

In [3]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])

stable_cols_present = [c for c in STABLE_COLS if c in train_raw.columns]
patient_sig = train_raw[stable_cols_present].astype(str).agg("|".join, axis=1).values
print(f"환자 추정 그룹(stable signature) 고유값 수: {len(set(patient_sig))} / 전체 {len(train_raw)}행")


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df_train = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
cat_cols = df_train.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    df_train[col] = le.fit_transform(df_train[col].astype(str))

y = df_train[TARGET_COL].values.astype(int)
X_base = df_train.drop(columns=[TARGET_COL])
global_mean = y.mean()
print(f"train shape: {X_base.shape}, 전체 평균 성공률: {global_mean:.4f}")


환자 추정 그룹(stable signature) 고유값 수: 3441 / 전체 256351행
train shape: (256351, 67), 전체 평균 성공률: 0.2583


## 2. 시작 파라미터 로드

In [4]:
def load_base_params(path: Path) -> dict:
    with open(path, encoding="utf-8") as f:
        loaded = json.load(f)
    return loaded.get("best_params", loaded) if isinstance(loaded, dict) else loaded

params_path = ROBUST_PARAMS_PATH if ROBUST_PARAMS_PATH.exists() else FALLBACK_PARAMS_PATH
base_params = load_base_params(params_path)
print(f"파라미터 출처: {params_path}")


파라미터 출처: ../lgbm_robust_best_params.json


## 3. 누수 없는 fold-safe target encoding 함수

In [5]:
def group_target_encode_fold(sig, y_arr, train_idx, val_idx, smoothing=TE_SMOOTHING):
    """train 부분에서만 그룹 평균을 계산 (leave-one-out). val 행은 그 train 그룹평균을
    그대로 받기만 함 — val 자신의 타깃 정보는 절대 사용 안 함."""
    g_mean = y_arr[train_idx].mean()

    sig_tr = pd.Series(sig[train_idx])
    y_tr = y_arr[train_idx]
    grp = pd.DataFrame({"sig": sig_tr, "y": y_tr}).groupby("sig")["y"].agg(["sum", "count"])

    sum_tr = sig_tr.map(grp["sum"]).values
    cnt_tr = sig_tr.map(grp["count"]).values
    te_train = (sum_tr - y_tr + smoothing * g_mean) / (cnt_tr - 1 + smoothing)

    sig_val = pd.Series(sig[val_idx])
    sum_val = sig_val.map(grp["sum"]).fillna(0).values
    cnt_val = sig_val.map(grp["count"]).fillna(0).values
    te_val = (sum_val + smoothing * g_mean) / (cnt_val + smoothing)

    return te_train, te_val


def evaluate_config(seeds, X, use_te):
    seed_aucs = []
    oof_per_seed = []
    for seed in seeds:
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        oof = np.zeros(len(y))
        for tr_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[tr_idx].copy(), X.iloc[val_idx].copy()
            y_tr = y[tr_idx]

            if use_te:
                te_tr, te_val = group_target_encode_fold(patient_sig, y, tr_idx, val_idx)
                X_tr["patient_group_te"] = te_tr
                X_val["patient_group_te"] = te_val

            m = LGBMClassifier(**base_params, random_state=seed, verbose=-1)
            m.fit(X_tr, y_tr)
            oof[val_idx] = m.predict_proba(X_val)[:, 1]

        seed_aucs.append(roc_auc_score(y, oof))
        oof_per_seed.append(oof)

    oof_per_seed = np.array(oof_per_seed)
    bagged_auc = roc_auc_score(y, oof_per_seed.mean(axis=0))
    return np.array(seed_aucs), bagged_auc


## 4. Stage 1 — 3시드 빠른 탐색

In [6]:
configs = [
    ("baseline", X_base, False),
    ("+ 환자그룹 target encoding", X_base, True),
]

print(f"Stage 1 탐색 (시드 {len(SEEDS_STAGE1)}개: {SEEDS_STAGE1})\n")
print(f"{'설정':<28} | {'평균AUC':>8} | {'표준편차':>8} | {'베이스라인差':>12} | 시간")
print("-" * 75)

stage1_results = {}
baseline_mean = None
for label, X_cfg, use_te in configs:
    t0 = time.time()
    aucs, _ = evaluate_config(SEEDS_STAGE1, X_cfg, use_te)
    mean_auc, std_auc = aucs.mean(), aucs.std()
    if label == "baseline":
        baseline_mean = mean_auc
    diff = mean_auc - baseline_mean if baseline_mean is not None else 0.0
    stage1_results[label] = {"X": X_cfg, "use_te": use_te, "mean": mean_auc, "std": std_auc}
    print(f"{label:<28} | {mean_auc:>8.5f} | {std_auc:>8.5f} | {diff:>+12.5f} | {time.time()-t0:>4.0f}s")


Stage 1 탐색 (시드 3개: [1004, 42, 7])

설정                           |    평균AUC |     표준편차 |       베이스라인差 | 시간
---------------------------------------------------------------------------
baseline                     |  0.73991 |  0.00005 |     +0.00000 |   51s
+ 환자그룹 target encoding       |  0.58105 |  0.00328 |     -0.15886 |   80s


## 5. Stage 2 — baseline보다 분명히 나으면 10시드 정밀검증

In [7]:
candidates = [
    label for label, r in stage1_results.items()
    if label != "baseline" and (r["mean"] - baseline_mean) > STAGE2_PROMOTION_THRESHOLD
]
print(f"Stage 2 정밀검증 후보: {candidates if candidates else '없음'}\n")

for label in ["baseline"] + candidates:
    r = stage1_results[label]
    aucs2, bagged2 = evaluate_config(SEEDS_STAGE2, r["X"], r["use_te"])
    print(f"[{label}] 10시드 평균={aucs2.mean():.5f} 표준편차={aucs2.std():.5f} 배깅AUC={bagged2:.5f}")

print()
print("비교 기준:")
print("  10시드 배깅 baseline: 0.74036 (기존) / 0.74037 (견고탐색)")
print("  노이즈 표준편차(참고): 0.00011")


Stage 2 정밀검증 후보: 없음

[baseline] 10시드 평균=0.73997 표준편차=0.00013 배깅AUC=0.74037

비교 기준:
  10시드 배깅 baseline: 0.74036 (기존) / 0.74037 (견고탐색)
  노이즈 표준편차(참고): 0.00011


## 6. 해석 가이드

- 효과가 거의 없다면(+0.0001 이하): `총 임신 횟수`/`총 출산 횟수` 같은 기존 컬럼이 이미 같은 정보를
  충분히 담고 있어서 재탕이었다는 뜻
- 의미있게 개선된다면: 그룹 기반 인코딩이 자기보고 카운트 컬럼과는 다른, 실제 기록 기반의 추가
  신호를 제공한다는 뜻 — `TE_SMOOTHING` 값을 줄여보거나(더 그룹별 평균에 가깝게), 그룹 정의를
  더 정교하게(예: `총 시술 횟수`까지 포함) 다듬어서 추가 개선 시도해볼 가치 있음
- 손해라면: 그룹 정의가 너무 coarse해서(같은 그룹 크기가 최대 26,577까지 갔던 것 기억) 진짜 같은
  환자가 아닌 행들끼리 노이즈를 주고받았을 가능성 — 이 경우 그룹을 더 좁히는 시도가 필요
